# TOURGO Analytics Dashboard

Data source: TourGo real data from S3/Delta Lake.

This notebook contains the dashboard sections used for the final demo. Day 3 fills Sections 1-3 with Silver and Gold Delta tables.

In [0]:
# == SECTION 1: BUSINESS OVERVIEW ==
from pyspark.sql.functions import col, sum as _sum, avg, round as _round

spark.sql("USE tourgo")

df_summary_bookings = spark.read.table("silver_bookings")
df_summary_revenues = spark.read.table("silver_revenues")
df_summary_users = spark.read.table("silver_users")
df_summary_tours = spark.read.table("silver_tours")
df_summary_reviews = spark.read.table("silver_reviews")

total_bookings = df_summary_bookings.count()
confirmed_bookings = df_summary_bookings.filter(col("status") == "confirmed").count()
total_revenue = df_summary_revenues.agg(_sum("total_amount").alias("total_revenue")).collect()[0]["total_revenue"] or 0
total_customers = df_summary_users.filter(col("role") == "CUSTOMER").count()
active_tours = df_summary_tours.filter(col("status").isin("approved", "active", "published")).count()
avg_rating = df_summary_reviews.agg(_round(avg("rating"), 2).alias("avg_rating")).collect()[0]["avg_rating"] or 0

print("=" * 60)
print("SECTION 1: TOURGO BUSINESS OVERVIEW")
print("=" * 60)
print(f"Total Revenue      : {total_revenue:>15,.0f} VND")
print(f"Total Bookings     : {total_bookings:>15,}")
print(f"Confirmed Bookings : {confirmed_bookings:>15,}")
print(f"Customers          : {total_customers:>15,}")
print(f"Active Tours       : {active_tours:>15,}")
print(f"Average Rating     : {avg_rating:>15}")

df_booking_status = df_summary_bookings.groupBy("status").count().orderBy(col("count").desc())
display(df_booking_status)
# Databricks chart suggestion: bar chart, X = status, Y = count.

SECTION 1: TOURGO BUSINESS OVERVIEW
Total Revenue      :     651,000,000 VND
Total Bookings     :             358
Confirmed Bookings :               0
Customers          :             250
Active Tours       :             205
Average Rating     :            4.42


status,count
CONFIRMED,319
CANCELLED,23
PENDING,16


In [0]:
# == SECTION 2: REVENUE ANALYSIS ==
df_rev_daily = spark.read.table("gold_revenue_daily")

print("=" * 60)
print("SECTION 2: REVENUE ANALYSIS")
print("=" * 60)
display(
    df_rev_daily
    .select("booking_date", "month", "total_revenue", "provider_revenue", "platform_fee", "num_transactions")
    .orderBy("booking_date")
)
# Databricks chart suggestion: line chart, X = booking_date, Y = total_revenue.

display(
    df_rev_daily
    .groupBy("month")
    .agg(
        _sum("total_revenue").alias("monthly_revenue"),
        _sum("provider_revenue").alias("monthly_provider_revenue"),
        _sum("platform_fee").alias("monthly_platform_fee"),
        _sum("num_transactions").alias("monthly_transactions")
    )
    .orderBy("month")
)
# Databricks chart suggestion: stacked/grouped bar chart by month.

SECTION 2: REVENUE ANALYSIS


booking_date,month,total_revenue,provider_revenue,platform_fee,num_transactions
2026-04-22,2026-04,1.57E7,1.413E7,1570000.0,3
2026-04-23,2026-04,3.81E7,3.429E7,3810000.0,6
2026-04-25,2026-04,3.43E7,3.087E7,3430000.0,5
2026-04-26,2026-04,6800000.0,6120000.0,680000.0,2
2026-04-28,2026-04,3.48E7,3.132E7,3480000.0,3
2026-04-29,2026-04,6600000.0,5940000.0,660000.0,1
2026-05-02,2026-05,1.09E7,9810000.0,1090000.0,3
2026-05-03,2026-05,4700000.0,4230000.0,470000.0,2
2026-05-04,2026-05,2.61E7,2.349E7,2610000.0,5
2026-05-05,2026-05,1.2E7,1.08E7,1200000.0,3


Databricks visualization. Run in Databricks to view.

month,monthly_revenue,monthly_provider_revenue,monthly_platform_fee,monthly_transactions
2026-04,1.363E8,1.2267E8,1.363E7,20
2026-05,5.147E8,4.6323E8,5.147E7,94


In [0]:
# == SECTION 3: PROVIDER AND TOUR PERFORMANCE ==
df_providers = spark.read.table("gold_provider_performance")
df_tours = spark.read.table("gold_tour_performance")

print("=" * 60)
print("SECTION 3A: TOP PROVIDERS")
print("=" * 60)
display(
    df_providers
    .select(
        "creator_id",
        "total_provider_revenue",
        "total_gmv",
        "total_confirmed_bookings",
        "active_tours_count",
        "avg_rating"
    )
    .orderBy(col("total_provider_revenue").desc())
    .limit(10)
)
# Databricks chart suggestion: bar chart, X = creator_id, Y = total_provider_revenue.

print("=" * 60)
print("SECTION 3B: TOP TOURS")
print("=" * 60)
display(
    df_tours
    .select(
        "tour_id",
        "title",
        "category_names",
        "total_revenue",
        "total_bookings",
        "total_travelers",
        "avg_rating",
        "review_count"
    )
    .orderBy(col("total_revenue").desc())
    .limit(10)
)
# Databricks chart suggestion: bar chart, X = title, Y = total_revenue.

SECTION 3A: TOP PROVIDERS


creator_id,total_provider_revenue,total_gmv,total_confirmed_bookings,active_tours_count,avg_rating
124,3.186E7,3.54E7,8,2,5.0
83,2.115E7,2.35E7,6,1,3.0
66,1.998E7,2.22E7,5,3,4.5
50,1.872E7,2.08E7,3,1,5.0
118,1.719E7,1.91E7,8,3,5.0
16,1.485E7,1.65E7,7,3,4.5
97,1.404E7,1.56E7,5,3,4.67
101,1.386E7,1.54E7,7,2,1.5
143,1.35E7,1.5E7,5,3,4.0
121,1.305E7,1.45E7,3,2,3.33


Databricks visualization. Run in Databricks to view.

SECTION 3B: TOP TOURS


tour_id,title,category_names,total_revenue,total_bookings,total_travelers,avg_rating,review_count
86,Nha Trang Biển Xanh Nắng Vàng v11,"Biển,Văn hóa,Khám phá",2.58E7,4,13,5.0,1
120,Huế Cố Đô Khám Phá Di Sản v15,"Ẩm thực,Nghỉ dưỡng",2.35E7,6,11,3.0,4
47,Đà Nẵng - Hội An Nghỉ Dưỡng v6,"Biển,Nghỉ dưỡng",2.08E7,3,5,5.0,1
96,Huế Cố Đô Khám Phá Di Sản v12,"Văn hóa,Ẩm thực,Sinh thái",1.4E7,4,8,5.0,1
28,Hội An Phố Cổ Huyền Ảo v4,Văn hóa,1.4E7,1,3,null,null
183,Đà Nẵng - Hội An Nghỉ Dưỡng v23,Khám phá,1.24E7,2,5,3.5,2
126,Nha Trang Biển Xanh Nắng Vàng v16,"Văn hóa,Nghỉ dưỡng",1.23E7,3,9,5.0,1
63,Đà Nẵng - Hội An Nghỉ Dưỡng v8,Sinh thái,1.2E7,1,4,4.0,1
108,Hội An Phố Cổ Huyền Ảo v14,"Ẩm thực,Khám phá,Sinh thái",1.2E7,2,5,null,null
55,Đà Nẵng - Hội An Nghỉ Dưỡng v7,Văn hóa,1.2E7,2,4,3.0,1


In [0]:
# Section 4: Payment Analysis - method split
# TODO: Analyze payment method, payment status, and total amount by method.
from pyspark.sql.functions import col, sum as _sum, round as _round

spark.sql("USE tourgo")

df_payment_method = spark.read.table("gold_payment_analysis")
df_payment_success = spark.read.table("gold_payment_success_rate")

total_payment_amount = df_payment_method.agg(_sum("total_amount").alias("total_amount")).collect()[0]["total_amount"] or 0
total_payment_count = df_payment_method.agg(_sum("count").alias("total_count")).collect()[0]["total_count"] or 0
avg_success_rate = df_payment_success.agg(_round(_sum(col("successful")) / _sum(col("total_attempts")) * 100, 2).alias("avg_success_rate")).collect()[0]["avg_success_rate"] or 0

print("=" * 60)
print("SECTION 4: PAYMENT ANALYSIS")
print("=" * 60)
print(f"Total Payment Amount : {total_payment_amount:>15,.0f} VND")
print(f"Payment Transactions : {total_payment_count:>15,.0f}")
print(f"Overall Success Rate : {avg_success_rate:>14.2f}%")

display(
    df_payment_method
    .select("payment_method", "status", "count", "total_amount")
    .orderBy("payment_method", "status")
)
# Databricks chart suggestion: pie chart, key = payment_method, value = count.

display(
    df_payment_success
    .select("payment_method", "total_attempts", "successful", "success_rate")
    .orderBy(col("success_rate").desc(), col("total_attempts").desc()))


SECTION 4: PAYMENT ANALYSIS
Total Payment Amount :   1,799,600,000 VND
Payment Transactions :             319
Overall Success Rate :          35.74%


payment_method,status,count,total_amount
BANK_TRANSFER,AWAITING_ADMIN,28,1.393E8
BANK_TRANSFER,FAILED,23,1.321E8
BANK_TRANSFER,SUCCESS,31,1.695E8
CASH,AWAITING_ADMIN,29,1.741E8
CASH,FAILED,21,1.286E8
CASH,SUCCESS,30,1.726E8
VIETQR,AWAITING_ADMIN,22,1.39E8
VIETQR,FAILED,25,1.446E8
VIETQR,SUCCESS,23,1.439E8
VNPAY,AWAITING_ADMIN,24,1.164E8


Databricks visualization. Run in Databricks to view.

payment_method,total_attempts,successful,success_rate
BANK_TRANSFER,82,31,37.8
CASH,80,30,37.5
VNPAY,87,30,34.48
VIETQR,70,23,32.86


In [0]:
# Section 5: Review and Rating
# Shows top-rated tours and overall rating distribution.
df_review_summary = spark.read.table("gold_review_summary")
df_rating_distribution = spark.read.table("gold_rating_distribution")

total_reviewed_tours = df_review_summary.count()
total_reviews = df_review_summary.agg(_sum("review_count").alias("total_reviews")).collect()[0]["total_reviews"] or 0
total_five_star = df_review_summary.agg(_sum("five_star").alias("five_star_reviews")).collect()[0]["five_star_reviews"] or 0

print("=" * 60)
print("SECTION 5: REVIEW AND RATING")
print("=" * 60)
print(f"Reviewed Tours       : {total_reviewed_tours:>15,}")
print(f"Total Reviews        : {total_reviews:>15,.0f}")
print(f"Five-star Reviews    : {total_five_star:>15,.0f}")

display(
    df_review_summary
    .select("tour_id", "title", "category_names", "avg_rating", "review_count", "five_star", "four_star", "one_star")
    .orderBy(col("avg_rating").desc(), col("review_count").desc())
    .limit(15)
)
# Databricks chart suggestion: bar chart, X = title, Y = avg_rating.

display(
    df_rating_distribution
    .select("rating", "count")
    .orderBy("rating"))


SECTION 5: REVIEW AND RATING
Reviewed Tours       :              86
Total Reviews        :             109
Five-star Reviews    :              74


tour_id,title,category_names,avg_rating,review_count,five_star,four_star,one_star
217,Tour Hạ Long Cao Cấp 2N1Đ v28,Ẩm thực,5.0,3,3,0,0
164,Hội An Phố Cổ Huyền Ảo v21,Khám phá,5.0,2,2,0,0
91,Phú Quốc Kỳ Thú 3N2Đ v12,"Văn hóa,Khám phá,Sinh thái",5.0,2,2,0,0
140,Hội An Phố Cổ Huyền Ảo v18,"Núi,Khám phá",5.0,2,2,0,0
74,Khám Phá Đà Lạt Mộng Mơ v10,"Văn hóa,Khám phá,Sinh thái",5.0,2,2,0,0
134,Nha Trang Biển Xanh Nắng Vàng v17,"Ẩm thực,Khám phá,Sinh thái",5.0,2,2,0,0
148,Hội An Phố Cổ Huyền Ảo v19,"Khám phá,Nghỉ dưỡng",5.0,2,2,0,0
64,Huế Cố Đô Khám Phá Di Sản v8,"Biển,Núi,Nghỉ dưỡng",5.0,2,2,0,0
208,Huế Cố Đô Khám Phá Di Sản v26,Núi,5.0,2,2,0,0
174,Nha Trang Biển Xanh Nắng Vàng v22,Văn hóa,5.0,1,1,0,0


Databricks visualization. Run in Databricks to view.

rating,count
1,2
2,6
3,10
4,17
5,74


In [0]:
# == SECTION 6: CUSTOMER SEGMENTS - K-MEANS RESULT ==
# Requires Day 6 visualization prep output tables:
#   - ml_customer_segments_pca
#   - ml_customer_cluster_profile

from pyspark.sql.functions import col, when, concat, lit, round as _round

spark.sql("USE tourgo")

df_segments_pca = spark.read.table("ml_customer_segments_pca")
df_cluster_profile = spark.read.table("ml_customer_cluster_profile")

cluster_name_expr = (
    when(col("cluster") == 0, "High-Risk / Unstable")
    .when(col("cluster") == 1, "Promising Newcomers")
    .when(col("cluster") == 2, "Dormant Users")
    .when(col("cluster") == 3, "VIP Loyalists")
    .otherwise(concat(lit("Cluster "), col("cluster")))
)

df_segments_labeled = df_segments_pca.withColumn("cluster_name", cluster_name_expr)
df_profile_labeled = df_cluster_profile.withColumn("cluster_name", cluster_name_expr)

total_segmented_customers = df_segments_labeled.count()
num_clusters = df_segments_labeled.select("cluster").distinct().count()

print("=" * 60)
print("SECTION 6: CUSTOMER SEGMENTS")
print("=" * 60)
print(f"Segmented Customers : {total_segmented_customers:>15,}")
print(f"Number of Clusters  : {num_clusters:>15,}")
print("Model              : K-Means, K=4")

display(
    df_segments_labeled
    .select(
        "user_id",
        "cluster",
        "cluster_name",
        "pc1",
        "pc2",
        "pca_x",
        "pca_y",
        "total_spent",
        "total_bookings",
        "cancellation_rate",
        "avg_rating_given"
    )
    .orderBy("cluster", "user_id")
)
# Databricks chart suggestion: scatter plot, X = pc1, Y = pc2, color/group = cluster_name.

display(
    df_profile_labeled
    .select(
        "cluster",
        "cluster_name",
        "customers",
        "avg_total_bookings",
        "avg_confirmed_bookings",
        "avg_total_spent",
        "avg_cancellation_rate",
        "avg_rating_given",
        "avg_unique_tours"
    )
    .orderBy("cluster")
)
# Databricks chart suggestion: bar chart, X = cluster_name, Y = avg_total_spent.

SECTION 6: CUSTOMER SEGMENTS
Segmented Customers :             250
Number of Clusters  :               4
Model              : K-Means, K=4


user_id,cluster,cluster_name,pc1,pc2,pca_x,pca_y,total_spent,total_bookings,cancellation_rate,avg_rating_given
212,0,High-Risk / Unstable,-3.0412720726173235,0.08180028227535094,-3.0412720726173235,0.08180028227535094,2.1E7,2,0.0,3.5
213,0,High-Risk / Unstable,-1.354349350291041,2.1322052998248817,-1.354349350291041,2.1322052998248817,8800000.0,2,0.5,5.0
217,0,High-Risk / Unstable,-2.127027703618941,-0.5426613494969117,-2.127027703618941,-0.5426613494969117,2000000.0,2,0.0,5.0
219,0,High-Risk / Unstable,-2.0212629902376165,-0.014779444277382816,-2.0212629902376165,-0.014779444277382816,5000000.0,2,0.0,4.0
227,0,High-Risk / Unstable,-2.587511700441886,-0.5078006774026045,-2.587511700441886,-0.5078006774026045,9000000.0,2,0.0,5.0
229,0,High-Risk / Unstable,-2.565315153202196,-0.3257451678982421,-2.565315153202196,-0.3257451678982421,1.0E7,2,0.0,5.0
230,0,High-Risk / Unstable,-2.159171729394025,-0.4699759483251577,-2.159171729394025,-0.4699759483251577,3000000.0,2,0.0,5.0
231,0,High-Risk / Unstable,-2.026512603392256,-0.6151187462853472,-2.026512603392256,-0.6151187462853472,0.0,2,0.0,5.0
235,0,High-Risk / Unstable,-2.2224696633280203,-0.14691133298937734,-2.2224696633280203,-0.14691133298937734,8400000.0,2,0.0,2.0
236,0,High-Risk / Unstable,-1.54124501782031,2.4154731604497353,-1.54124501782031,2.4154731604497353,1.36E7,2,0.5,5.0


cluster,cluster_name,customers,avg_total_bookings,avg_confirmed_bookings,avg_total_spent,avg_cancellation_rate,avg_rating_given,avg_unique_tours
0,High-Risk / Unstable,57,2.0,1.89,5522807.0,0.018,4.18,2.0
1,Promising Newcomers,9,0.0,0.0,0.0,0.0,0.0,0.0
2,Dormant Users,60,2.0,1.67,1958333.0,0.117,0.39,1.98
3,VIP Loyalists,124,1.0,0.9,1763710.0,0.056,1.27,1.0


In [0]:
# == SECTION 7: REVENUE FORECAST ==
# Day 7 [C]: forecast chart (plan layout Section 6)
# Requires table: ml_revenue_forecast (from 04b_ml_forecasting.ipynb)

from pyspark.sql.functions import col, when, abs as _abs, round as _round, avg

spark.sql("USE tourgo")

df_forecast = spark.read.table("ml_revenue_forecast")

df_forecast_viz = df_forecast \
    .withColumn("predicted", when(col("predicted") < 0, 0).otherwise(col("predicted"))) \
    .withColumn("absolute_error", _abs(col("actual") - col("predicted"))) \
    .withColumn(
        "error_pct",
        _round(when(col("actual") > 0, col("absolute_error") / col("actual") * 100).otherwise(0), 2)
    ) \
    .orderBy("booking_date")

forecast_days = df_forecast_viz.count()
avg_abs_error = df_forecast_viz.agg(_round(avg("absolute_error"), 0).alias("avg_abs_error")).collect()[0]["avg_abs_error"] or 0

print("=" * 60)
print("SECTION 7: REVENUE FORECAST")
print("=" * 60)
print("Model       : Linear Regression")
print("R2          : 0.8790")
print("RMSE        : 4,516,570 VND")
print(f"Forecast Days: {forecast_days:>10,}")
print(f"Avg Abs Err : {avg_abs_error:>10,.0f} VND")

display(
    df_forecast_viz.select(
        "booking_date",
        "actual",
        "predicted",
        "absolute_error",
        "error_pct"
    )
)
# Databricks chart suggestion: line chart, X = booking_date, Y = actual and predicted.

SECTION 7: REVENUE FORECAST
Model       : Linear Regression
R2          : 0.8790
RMSE        : 4,516,570 VND
Forecast Days:          7
Avg Abs Err :  3,411,943 VND


booking_date,actual,predicted,absolute_error,error_pct
2026-05-22,2.73E7,3.0674103717493277E7,3374103.717493277,12.36
2026-05-23,9100000.0,1.4841881972803582E7,5741881.9728035815,63.1
2026-05-24,3.29E7,3.122954248046785E7,1670457.5195321515,5.08
2026-05-25,3.78E7,3.740216598562412E7,397834.01437588036,1.05
2026-05-26,5100000.0,0.0,5100000.0,100.0
2026-05-27,7600000.0,8688900.300107382,1088900.3001073822,14.33
2026-05-28,6800000.0,289573.53479719535,6510426.465202805,95.74


In [0]:
# == SECTION 8: ANOMALY ALERTS ==
# Day 8 [C]: visualization only — input from 04c_anomaly_detection.ipynb
# Requires table: gold_anomalies

from pyspark.sql.functions import col, count as _count, when, lit, concat, round as _round

spark.sql("USE tourgo")

df_anom = spark.read.table("gold_anomalies")

# Enrich tour_id with title for readable alerts
df_tours = spark.read.table("silver_tours")
df_anom_viz = df_anom.join(
    df_tours.select(col("id").alias("tour_id"), col("title").alias("tour_title")),
    on="tour_id",
    how="left",
)

severity_rank = (
    when(col("severity") == "HIGH", lit(1))
    .when(col("severity") == "MEDIUM", lit(2))
    .when(col("severity") == "LOW", lit(3))
    .otherwise(lit(4))
)

df_anom_viz = (
    df_anom_viz
    .withColumn("_severity_rank", severity_rank)
    .withColumn(
        "value_display",
        when(col("anomaly_type").contains("hủy"), _round(col("value") * 100, 1))
        .otherwise(col("value"))
    )
    .orderBy("_severity_rank", col("date").desc_nulls_last())
)

total_anomalies = df_anom_viz.count()
high_count = df_anom_viz.filter(col("severity") == "HIGH").count()
medium_count = df_anom_viz.filter(col("severity") == "MEDIUM").count()
low_count = df_anom_viz.filter(col("severity") == "LOW").count()

print("=" * 60)
print("SECTION 8: ANOMALY ALERTS")
print("=" * 60)
print(f"Total Anomalies : {total_anomalies:>15,}")
print(f"HIGH Severity   : {high_count:>15,}")
print(f"MEDIUM Severity : {medium_count:>15,}")
print(f"LOW Severity    : {low_count:>15,}")
print("Source          : 04c_anomaly_detection (Z-score / statistical)")

display(
    df_anom_viz.select(
        "date",
        "anomaly_type",
        "severity",
        "user_id",
        "tour_id",
        "tour_title",
        "value_display",
    ).drop("_severity_rank")
)
# Databricks chart suggestion: table view; color by severity (HIGH=red, MEDIUM=yellow).

display(
    df_anom_viz.groupBy("severity", "_severity_rank")
    .agg(_count(lit(1)).alias("alert_count"))
    .orderBy("_severity_rank")
    .select("severity", "alert_count")
)
# Databricks chart suggestion: bar chart, X = severity, Y = alert_count.

display(
    df_anom_viz.groupBy("anomaly_type")
    .agg(_count(lit(1)).alias("alert_count"))
    .orderBy(col("alert_count").desc())
)
# Databricks chart suggestion: bar chart, X = anomaly_type, Y = alert_count.

SECTION 8: ANOMALY ALERTS
Total Anomalies :              11
HIGH Severity   :               2
MEDIUM Severity :               9
LOW Severity    :               0
Source          : 04c_anomaly_detection (Z-score / statistical)


date,anomaly_type,severity,user_id,tour_id,tour_title,value_display
2026-05-10,User booking bất thường trong 1 ngày,HIGH,340,null,null,2.0
2026-04-25,User booking bất thường trong 1 ngày,HIGH,328,null,null,2.0
2026-05-09,Revenue đột biến tăng,MEDIUM,null,null,null,4.93E7
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,39,Đà Nẵng - Hội An Nghỉ Dưỡng v5,33.3
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,48,Huế Cố Đô Khám Phá Di Sản v6,33.3
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,228,Hội An Phố Cổ Huyền Ảo v29,33.3
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,2,Khám Phá Đà Lạt Mộng Mơ v1,33.3
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,159,Đà Nẵng - Hội An Nghỉ Dưỡng v20,33.3
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,15,Đà Nẵng - Hội An Nghỉ Dưỡng v2,33.3
null,Tour có tỷ lệ hủy bất thường,MEDIUM,null,129,Tour Hạ Long Cao Cấp 2N1Đ v17,33.3


severity,alert_count
HIGH,2
MEDIUM,9


anomaly_type,alert_count
Tour có tỷ lệ hủy bất thường,8
User booking bất thường trong 1 ngày,2
Revenue đột biến tăng,1


In [0]:
# == SECTION 9: REAL-TIME STREAMING STATUS ==
# Day 9 [C]: visualization only — input from 06_streaming.ipynb
# Requires table: stream_bookings_bronze (after [D] simulator + [B] streaming notebook)

from pyspark.sql.functions import col, sum as _sum, count as _count, lit, max as _max

spark.sql("USE tourgo")

try:
    df_stream_data = spark.read.table("stream_bookings_bronze")
except Exception as exc:
    raise RuntimeError(
        "Chưa có bảng stream_bookings_bronze. "
        "Yêu cầu [D] chạy streaming_simulator.py, sau đó [B] chạy 06_streaming.ipynb."
    ) from exc

df_tours = spark.read.table("silver_tours")
df_stream_viz = df_stream_data.join(
    df_tours.select(col("id").alias("tour_id"), col("title").alias("tour_title")),
    on="tour_id",
    how="left",
)

stream_count = df_stream_viz.count()
stream_revenue = (
    df_stream_viz.filter(col("status") == "confirmed")
    .agg(_sum("total_price").alias("revenue"))
    .collect()[0]["revenue"]
    or 0
)
confirmed_count = df_stream_viz.filter(col("status") == "confirmed").count()
pending_count = df_stream_viz.filter(col("status") == "pending").count()

latest_processed = None
if "processed_at" in df_stream_viz.columns:
    latest_processed = df_stream_viz.agg(_max("processed_at").alias("ts")).collect()[0]["ts"]

print("=" * 60)
print("SECTION 9: REAL-TIME STREAMING STATUS")
print("=" * 60)
print(f"New Bookings Received : {stream_count:>15,}")
print(f"Confirmed Bookings    : {confirmed_count:>15,}")
print(f"Pending Bookings      : {pending_count:>15,}")
print(f"Streaming Revenue     : {stream_revenue:>15,.0f} VND")
print(f"Last Processed At     : {latest_processed}")
print("Source                : S3 -> 06_streaming -> stream_bookings_bronze")
print("Trigger               : Every ~30 seconds (simulator)")

display(
    df_stream_viz.groupBy("status")
    .agg(_count(lit(1)).alias("booking_count"))
    .orderBy(col("booking_count").desc())
)
# Databricks chart suggestion: bar or pie chart, key = status, value = booking_count.

display(
    df_stream_viz.select(
        "processed_at",
        "id",
        "user_id",
        "tour_id",
        "tour_title",
        "number_of_people",
        "total_price",
        "status",
        "booking_date",
        "source",
        "created_at",
    )
    .orderBy(col("processed_at").desc_nulls_last())
    .limit(10)
)
# Databricks chart suggestion: table view — 10 bookings mới nhất.

if stream_count >= 2 and "processed_at" in df_stream_viz.columns:
    display(
        df_stream_viz.groupBy("processed_at")
        .agg(_count(lit(1)).alias("bookings_in_batch"))
        .orderBy("processed_at")
    )
    # Databricks chart suggestion: line chart, X = processed_at, Y = bookings_in_batch.

SECTION 9: REAL-TIME STREAMING STATUS
New Bookings Received :               2
Confirmed Bookings    :               1
Pending Bookings      :               1
Streaming Revenue     :       1,500,000 VND
Last Processed At     : 2026-06-03 08:17:09.755736
Source                : S3 -> 06_streaming -> stream_bookings_bronze
Trigger               : Every ~30 seconds (simulator)


status,booking_count
pending,1
confirmed,1


processed_at,id,user_id,tour_id,tour_title,number_of_people,total_price,status,booking_date,source,created_at
2026-06-03T08:17:09.755Z,1780072869198,26,1,Tour Hạ Long Cao Cấp 2N1Đ v1,4,1500000.0,confirmed,2026-05-29,streaming_simulator,2026-05-29 23:41:08
2026-06-03T08:17:09.755Z,1780072869064,11,2,Khám Phá Đà Lạt Mộng Mơ v1,3,2000000.0,pending,2026-05-29,streaming_simulator,2026-05-29 23:41:08


processed_at,bookings_in_batch
2026-06-03T08:17:09.755Z,2


In [0]:
# == SECTION 10: TOUR RECOMMENDATIONS ==
# Day 7 output — visualization only (built in ML pipeline, not here)
# Requires table: ml_tour_recommendations

from pyspark.sql.functions import col, when, concat, lit

spark.sql("USE tourgo")

df_reco_raw = spark.read.table("ml_tour_recommendations")

if "cluster" in df_reco_raw.columns:
    df_reco = df_reco_raw
elif "prediction" in df_reco_raw.columns:
    df_reco = df_reco_raw.withColumn("cluster", col("prediction"))
else:
    raise ValueError("ml_tour_recommendations must contain either 'cluster' or 'prediction'.")

cluster_name_expr = (
    when(col("cluster") == 0, "High-Risk / Unstable")
    .when(col("cluster") == 1, "Promising Newcomers")
    .when(col("cluster") == 2, "Dormant Users")
    .when(col("cluster") == 3, "VIP Loyalists")
    .otherwise(concat(lit("Cluster "), col("cluster")))
)

df_reco_labeled = df_reco.withColumn("cluster_name", cluster_name_expr)

reco_rows = df_reco_labeled.count()
reco_clusters = df_reco_labeled.select("cluster").distinct().count()

print("=" * 60)
print("SECTION 10: TOUR RECOMMENDATIONS")
print("=" * 60)
print(f"Recommendation Rows : {reco_rows:>15,}")
print(f"Covered Clusters    : {reco_clusters:>15,}")

reco_display_cols = [
    c
    for c in [
        "cluster",
        "cluster_name",
        "rank",
        "title",
        "price",
        "avg_rating",
        "category_names",
        "bookings_by_cluster",
    ]
    if c in df_reco_labeled.columns
]
reco_order_cols = [c for c in ["cluster", "rank"] if c in df_reco_labeled.columns]

display(df_reco_labeled.select(*reco_display_cols).orderBy(*reco_order_cols))
# Databricks chart suggestion: table view, grouped by cluster_name and ordered by rank.

SECTION 10: TOUR RECOMMENDATIONS
Recommendation Rows :              23
Covered Clusters    :               3


cluster,cluster_name,rank,title,price,avg_rating,category_names,bookings_by_cluster
0,High-Risk / Unstable,1,Hội An Phố Cổ Huyền Ảo v18,2000000.0,5.0,"Núi,Khám phá",2
0,High-Risk / Unstable,1,Hội An Phố Cổ Huyền Ảo v30,2000000.0,5.0,Biển,2
0,High-Risk / Unstable,1,Huế Cố Đô Khám Phá Di Sản v12,1700000.0,5.0,"Văn hóa,Ẩm thực,Sinh thái",2
0,High-Risk / Unstable,1,Nha Trang Biển Xanh Nắng Vàng v11,2200000.0,5.0,"Biển,Văn hóa,Khám phá",2
0,High-Risk / Unstable,1,Tour Hạ Long Cao Cấp 2N1Đ v28,2500000.0,5.0,Ẩm thực,2
0,High-Risk / Unstable,1,Hội An Phố Cổ Huyền Ảo v23,2000000.0,5.0,Khám phá,2
0,High-Risk / Unstable,1,Phú Quốc Kỳ Thú 3N2Đ v24,3500000.0,5.0,"Biển,Văn hóa,Nghỉ dưỡng",2
0,High-Risk / Unstable,1,Khám Phá Đà Lạt Mộng Mơ v10,1800000.0,5.0,"Văn hóa,Khám phá,Sinh thái",2
0,High-Risk / Unstable,1,Nha Trang Biển Xanh Nắng Vàng v17,2200000.0,5.0,"Ẩm thực,Khám phá,Sinh thái",2
0,High-Risk / Unstable,1,Huế Cố Đô Khám Phá Di Sản v21,1700000.0,5.0,"Biển,Ẩm thực",2
